In [195]:
import os
import requests
import json
from dotenv import load_dotenv
# from langchain_openai import ChatOpenAI
from langchain_community.utilities.sql_database import SQLDatabase
from langchain_experimental.sql import SQLDatabaseSequentialChain
from langchain_community.utilities.sql_database import SQLDatabase
from langchain_core.prompts import MessagesPlaceholder, ChatPromptTemplate
from langchain_core.prompts import PromptTemplate
from typing_extensions import Annotated, TypedDict
from langchain_community.tools.sql_database.tool import QuerySQLDataBaseTool
from langchain_community.utilities.sql_database import SQLDatabase
from sqlalchemy import create_engine
# from langgraph.prebuilt import create_react_agent
from langgraph_supervisor import create_supervisor
from langchain_openai import ChatOpenAI
import psycopg2
import time 
import yaml
import json 
from langchain_community.agent_toolkits import JsonToolkit, create_json_agent
from langchain_community.tools.json.tool import JsonSpec
from langchain_openai import OpenAI
from langchain.tools import Tool

### Add Thingsboard details

In [196]:
THINGSBOARD_URL = "http://localhost:8080"
USERNAME = "tenant@thingsboard.org"
PASSWORD = "tenant"
DASHBOARD_ID = "http://localhost:8080/tenants"

DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "thingsboard"
DB_USER = "thingsboard"
DB_PASSWORD = "postgres"


THINGSBOARD_HOST = "http://localhost:8080"
USERNAME = "tenant@thingsboard.org"
PASSWORD = "tenant"

# Authenticate and get the JWT token
auth_url = f'{THINGSBOARD_HOST}/api/auth/login'
auth_payload = {'username': USERNAME, 'password': PASSWORD}
auth_response = requests.post(auth_url, json=auth_payload)
auth_response.raise_for_status()
jwt_token = auth_response.json()['token']

In [197]:
# 2. Get device ID by name
def get_device_id_by_name(device_name, token):
    headers = {
        "Content-Type": "application/json",
        "X-Authorization": f"Bearer {token}"
    }
    url = f"{THINGSBOARD_URL}/api/tenant/devices?deviceName={device_name}"
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    device = response.json()
    return device['id']['id'] if device else None

In [198]:
def get_device_keys(jwt_token, device_id):
    url = f"{THINGSBOARD_URL}/api/plugins/telemetry/DEVICE/{device_id}/keys/timeseries"
    headers = {
        "X-Authorization": f"Bearer {jwt_token}"
    }

    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        return response.json()  # Returns a list of key names
    else:
        return {
            "error": f"Failed to fetch keys: {response.status_code}",
            "details": response.text
        }

In [202]:
# change the device name 
device_name = "HUM-0100"
# device_name = "TEMP-0200"
sensor_id = get_device_id_by_name(device_name, jwt_token)
device_id = sensor_id
sensor_id

'e7fe7880-1221-11f0-a236-5f0808fc8cdf'

In [203]:
device_key = get_device_keys(jwt_token, sensor_id)
device_key = device_key[-1] # ['deviceId', 'unit', 'relative_humidity']
device_key 

'relative_humidity'

# Get data for the last 24 hours

In [204]:
# Last 24 hours timestamps
end_ts = int(time.time() * 1000)  
start_ts = end_ts - (340 * 60 * 60 * 1000)  # 24 hours ago

# keys = "temperature"  
keys = device_key


def get_historical_data(jwt_token, device_id, start_ts, end_ts, keys):
    url = f"{THINGSBOARD_URL}/api/plugins/telemetry/DEVICE/{device_id}/values/timeseries"
    params = {"keys": keys, "startTs": start_ts, "endTs": end_ts, "limit": 100}
    headers = {"X-Authorization": f"Bearer {jwt_token}"}
    
    response = requests.get(url, headers=headers, params=params)
    return response.json() if response.status_code == 200 else {"error": "Failed to fetch telemetry data"}

response = get_historical_data(jwt_token, device_id, start_ts, end_ts, keys)
response

{'relative_humidity': [{'ts': 1743868982145, 'value': '81.49146754982857'},
  {'ts': 1743868377678, 'value': '84.26805694586591'},
  {'ts': 1743867773300, 'value': '88.75880788898665'},
  {'ts': 1743867169530, 'value': '86.1493865122688'},
  {'ts': 1743866565125, 'value': '88.42744643480691'},
  {'ts': 1743865960772, 'value': '83.37206111868072'},
  {'ts': 1743863808237, 'value': '82.40490380856446'}]}

# Debugging devices 

In [205]:
uuid_list = [
    "e799ea50-1221-11f0-a236-5f0808fc8cdf",
    "e7a97ab0-1221-11f0-a236-5f0808fc8cdf",
    "e7acaf00-1221-11f0-a236-5f0808fc8cdf",
    "e7af6e20-1221-11f0-a236-5f0808fc8cdf",
    "e7b27b60-1221-11f0-a236-5f0808fc8cdf",
    "e7b4c550-1221-11f0-a236-5f0808fc8cdf",
    "e7b67300-1221-11f0-a236-5f0808fc8cdf",
    "e7b86ed0-1221-11f0-a236-5f0808fc8cdf",
    "e7ba6aa0-1221-11f0-a236-5f0808fc8cdf",
    "e7bc1850-1221-11f0-a236-5f0808fc8cdf",
    "e7bded10-1221-11f0-a236-5f0808fc8cdf",
    "e7bfc1d0-1221-11f0-a236-5f0808fc8cdf",
    "e7c14870-1221-11f0-a236-5f0808fc8cdf",
    "e7c2f620-1221-11f0-a236-5f0808fc8cdf",
    "e7c51900-1221-11f0-a236-5f0808fc8cdf",
    "e7c6edc0-1221-11f0-a236-5f0808fc8cdf",
    "e7c89b70-1221-11f0-a236-5f0808fc8cdf",
    "e7ca7030-1221-11f0-a236-5f0808fc8cdf",
    "e7cc9310-1221-11f0-a236-5f0808fc8cdf",
    "e7cedd00-1221-11f0-a236-5f0808fc8cdf",
    "e7d0b1c0-1221-11f0-a236-5f0808fc8cdf",
    "e7d349d0-1221-11f0-a236-5f0808fc8cdf",
    "e7d1ea40-1221-11f0-a236-5f0808fc8cdf",
    "e7d65710-1221-11f0-a236-5f0808fc8cdf",
    "e7da0090-1221-11f0-a236-5f0808fc8cdf",
    "e7ddaa10-1221-11f0-a236-5f0808fc8cdf",
    "e7e1c8c0-1221-11f0-a236-5f0808fc8cdf",
    "e7e460d0-1221-11f0-a236-5f0808fc8cdf",
    "e7e5e770-1221-11f0-a236-5f0808fc8cdf",
    "e7eaa260-1221-11f0-a236-5f0808fc8cdf",
    "e7ee4be0-1221-11f0-a236-5f0808fc8cdf",
    "e7f1f560-1221-11f0-a236-5f0808fc8cdf",
    "e7f550c0-1221-11f0-a236-5f0808fc8cdf",
    "e7f8d330-1221-11f0-a236-5f0808fc8cdf",
    "e7fc2e90-1221-11f0-a236-5f0808fc8cdf",
    "e8072b10-1221-11f0-a236-5f0808fc8cdf",
    "e80be600-1221-11f0-a236-5f0808fc8cdf",
    "e80cf770-1221-11f0-a236-5f0808fc8cdf",
    "e81079e0-1221-11f0-a236-5f0808fc8cdf",
    "e813ae30-1221-11f0-a236-5f0808fc8cdf",
    "e8164640-1221-11f0-a236-5f0808fc8cdf",
    "e819efc0-1221-11f0-a236-5f0808fc8cdf",
    "e81eaab0-1221-11f0-a236-5f0808fc8cdf",
    "e8249e20-1221-11f0-a236-5f0808fc8cdf",
    "e828bcd0-1221-11f0-a236-5f0808fc8cdf",
    "e7d4a960-1221-11f0-a236-5f0808fc8cdf",
    "e7d78f90-1221-11f0-a236-5f0808fc8cdf",
    "e7d8c810-1221-11f0-a236-5f0808fc8cdf",
    "e7db3910-1221-11f0-a236-5f0808fc8cdf",
    "e7dc7190-1221-11f0-a236-5f0808fc8cdf",
    "e7e06930-1221-11f0-a236-5f0808fc8cdf",
    "e7e76e10-1221-11f0-a236-5f0808fc8cdf",
    "e7ebdae0-1221-11f0-a236-5f0808fc8cdf",
    "e7ef8460-1221-11f0-a236-5f0808fc8cdf",
    "e7f306d0-1221-11f0-a236-5f0808fc8cdf",
    "e7f41840-1221-11f0-a236-5f0808fc8cdf",
    "e7f66230-1221-11f0-a236-5f0808fc8cdf",
    "e7f79ab0-1221-11f0-a236-5f0808fc8cdf",
    "e7f9e4a0-1221-11f0-a236-5f0808fc8cdf",
    "e7fb4430-1221-11f0-a236-5f0808fc8cdf",
    "e7fd6710-1221-11f0-a236-5f0808fc8cdf",
    "e7fe7880-1221-11f0-a236-5f0808fc8cdf",
    "e7ff89f0-1221-11f0-a236-5f0808fc8cdf",
    "e8009b60-1221-11f0-a236-5f0808fc8cdf",
    "e801acd0-1221-11f0-a236-5f0808fc8cdf",
    "e802be40-1221-11f0-a236-5f0808fc8cdf",
    "e803cfb0-1221-11f0-a236-5f0808fc8cdf",
    "e8050830-1221-11f0-a236-5f0808fc8cdf",
    "e80619a0-1221-11f0-a236-5f0808fc8cdf",
    "e7df09a0-1221-11f0-a236-5f0808fc8cdf",
    "e7e30140-1221-11f0-a236-5f0808fc8cdf",
    "e8088aa0-1221-11f0-a236-5f0808fc8cdf",
    "e80ef340-1221-11f0-a236-5f0808fc8cdf",
    "e81275b0-1221-11f0-a236-5f0808fc8cdf",
    "e8150dc0-1221-11f0-a236-5f0808fc8cdf",
    "e8205860-1221-11f0-a236-5f0808fc8cdf",
    "e82624c0-1221-11f0-a236-5f0808fc8cdf",
    "e829a730-1221-11f0-a236-5f0808fc8cdf",
    "e82e8930-1221-11f0-a236-5f0808fc8cdf",
    "e8340770-1221-11f0-a236-5f0808fc8cdf",
    "e8384d30-1221-11f0-a236-5f0808fc8cdf",
    "e8395ea0-1221-11f0-a236-5f0808fc8cdf",
    "e83b3360-1221-11f0-a236-5f0808fc8cdf",
    "e8419c00-1221-11f0-a236-5f0808fc8cdf",
    "e84370c0-1221-11f0-a236-5f0808fc8cdf",
    "e7e91bc0-1221-11f0-a236-5f0808fc8cdf",
    "e7ed3a70-1221-11f0-a236-5f0808fc8cdf",
    "e7f0bce0-1221-11f0-a236-5f0808fc8cdf",
    "e8116440-1221-11f0-a236-5f0808fc8cdf",
    "e81757b0-1221-11f0-a236-5f0808fc8cdf",
    "e8190560-1221-11f0-a236-5f0808fc8cdf",
    "e81b2840-1221-11f0-a236-5f0808fc8cdf",
    "e81d7230-1221-11f0-a236-5f0808fc8cdf",
    "e82190e0-1221-11f0-a236-5f0808fc8cdf",
    "e8227b40-1221-11f0-a236-5f0808fc8cdf",
    "e8238cb0-1221-11f0-a236-5f0808fc8cdf",
    "e8278450-1221-11f0-a236-5f0808fc8cdf",
    "e82a9190-1221-11f0-a236-5f0808fc8cdf",
    "e82c8d60-1221-11f0-a236-5f0808fc8cdf",
    "e82fc1b0-1221-11f0-a236-5f0808fc8cdf",
    "e831e490-1221-11f0-a236-5f0808fc8cdf",
    "e83518e0-1221-11f0-a236-5f0808fc8cdf",
    "e83762d0-1221-11f0-a236-5f0808fc8cdf",
    "e83a7010-1221-11f0-a236-5f0808fc8cdf",
    "e83c6be0-1221-11f0-a236-5f0808fc8cdf",
    "e83d7d50-1221-11f0-a236-5f0808fc8cdf",
    "e83eb5d0-1221-11f0-a236-5f0808fc8cdf",
    "e83fee50-1221-11f0-a236-5f0808fc8cdf",
    "e8448230-1221-11f0-a236-5f0808fc8cdf",
    "e82b7bf0-1221-11f0-a236-5f0808fc8cdf",
    "e82d9ed0-1221-11f0-a236-5f0808fc8cdf",
    "e830d320-1221-11f0-a236-5f0808fc8cdf",
    "e832cef0-1221-11f0-a236-5f0808fc8cdf",
    "e8362a50-1221-11f0-a236-5f0808fc8cdf",
    "d7f871e0-19f9-11f0-a236-5f0808fc8cdf"
]

In [ ]:
for device_id in uuid_list: 
    device_key = get_device_keys(jwt_token, device_id)
    print(device_key, device_id)